# 🏗️ 3D Asset Factory
### 🛠️ Developed by: **Arafat Ahmed Mubin**
### 🌐 Brand: **2M**
---
**Text → Concept Art (FLUX) → 3D Mesh (TripoSR) → .GLB Export**
Optimised with `device_map='auto'` for Colab & Kaggle T4 GPUs.

## 🛠️ Step 1: Environment Setup

In [ ]:
import os
try:
    import google.colab
    ENV = "Colab"
except ImportError:
    ENV = "Kaggle" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else "Local"
print(f"✅ Detected: {ENV}")
if ENV == "Kaggle":
    print("ℹ️  Ensure 'Internet' is ON in Kaggle Notebook Settings.")
!pip install -q -U diffusers transformers accelerate bitsandbytes gradio rembg trimesh
!pip install -q -U torchaudio
# TripoSR: install from GitHub (most reliable)
!pip install -q git+https://github.com/VAST-AI-Research/TripoSR.git
print("✅ All packages installed.")

## 🧠 Step 2: Load FLUX Concept Engine & TripoSR

In [ ]:
import torch, gc, os
import numpy as np
from PIL import Image
from diffusers import FluxPipeline
import trimesh, gradio as gr
from rembg import remove
from tsr.system import TSR

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
flux_pipe  = None
tripo_model = None

def clear_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def load_flux():
    global flux_pipe
    if flux_pipe is None:
        clear_memory()
        print("⏳ Loading FLUX.1-schnell...")
        flux_pipe = FluxPipeline.from_pretrained(
            "black-forest-labs/FLUX.1-schnell",
            torch_dtype=torch.float16,
            device_map="auto"
        )
        print("✅ FLUX ready.")
    return flux_pipe

def load_tripo():
    global tripo_model
    if tripo_model is None:
        clear_memory()
        print("⏳ Loading TripoSR...")
        tripo_model = TSR.from_pretrained(
            "stabilityai/TripoSR",
            weight_name="model.ckpt",
            config_name="config.yaml"
        )
        tripo_model.renderer.set_chunk_size(131072)
        tripo_model.to(DEVICE)
        print("✅ TripoSR ready.")
    return tripo_model

print("✅ Loader functions defined.")

## 🎛️ Step 3: Asset Generation Pipeline

In [ ]:
import uuid

def factory_process(prompt, decimation_ratio, image_input, use_image):
    if use_image and image_input is not None:
        concept_image = image_input.convert("RGB")
    elif prompt.strip():
        f_pipe = load_flux()
        concept_prompt = f"{prompt}, isolated on pure white background, high quality concept art, 3D game asset"
        print("🎨 Generating concept art...")
        concept_image = f_pipe(
            concept_prompt,
            num_inference_steps=4,
            width=512, height=512,
            max_sequence_length=256
        ).images[0]
    else:
        return None, None, None, "❌ Provide a text prompt or upload an image."

    print("✂️ Removing background...")
    image_no_bg = remove(concept_image)
    # Resize to 512x512 for TripoSR
    image_no_bg = image_no_bg.resize((512, 512))

    t_model = load_tripo()
    print("🏗️ Running TripoSR 3D reconstruction...")
    with torch.no_grad():
        scene_codes = t_model([image_no_bg], device=DEVICE)

    print("🔧 Extracting mesh...")
    meshes = t_model.extract_mesh(scene_codes, resolution=256)
    mesh   = meshes[0]

    if decimation_ratio < 1.0:
        target = int(len(mesh.faces) * decimation_ratio)
        mesh = mesh.simplify_quadratic_decimation(target)

    uid = uuid.uuid4().hex[:8]
    glb_path = f"asset_{uid}.glb"
    mesh.export(glb_path)
    clear_memory()
    print(f"✅ Exported {glb_path}")
    return concept_image, glb_path, glb_path, f"✅ Saved as {glb_path}"

print("✅ Pipeline function ready.")

## 🏗️ Step 4: Asset Factory Dashboard

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(), title="2M 3D Asset Factory") as factory:
    gr.Markdown("# 🏗️ 2M 3D Asset Factory\n**FLUX Concept Art → TripoSR 3D → .GLB Export · T4 Optimized**")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Input")
            use_image = gr.Checkbox(label="Upload my own image instead of using text", value=False)
            img_input = gr.Image(type="pil", label="Upload Image (optional)", visible=False)
            use_image.change(lambda x: gr.update(visible=x), use_image, img_input)

            prompt_in = gr.Textbox(label="Text Prompt", placeholder="A robotic knight in silver armor, detailed sci-fi style...")
            decimate  = gr.Slider(0.1, 1.0, value=0.5, step=0.1, label="Mesh Density (lower = more optimized)")
            gen_btn   = gr.Button("⚒️ Forge 3D Asset", variant="primary")
            status    = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=2):
            gr.Markdown("### Output")
            with gr.Row():
                img_out  = gr.Image(label="Concept Art")
                mesh_out = gr.Model3D(label="3D Preview (.glb)")
            dl_btn = gr.File(label="📥 Download GLB")

    gen_btn.click(
        factory_process,
        inputs=[prompt_in, decimate, img_input, use_image],
        outputs=[img_out, mesh_out, dl_btn, status]
    )

factory.launch(share=True, debug=False)

---
**© 2026 2M | All Rights Reserved**  
*Powered by the 2M Ecosystem (2M Dev's · 2M Future Facts · 2M Business Blogs)*